In [2]:
import gzip
import shutil
import requests
import pandas as pd 
from lxml import etree

In [2]:
url = 'https://dblp.uni-trier.de/xml/dblp.xml.gz'
dblp_gz = '../data/raw/dblp.xml.gz'
dblp_xml = '../data/raw/dblp.xml'

print('*** Downloading DBLP file...')
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(dblp_gz, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
print('***File downloaded:', dblp_gz)

print('***Unboxing file...')
with gzip.open(dblp_gz, 'rb') as f_in:
    with open(dblp_xml, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
print('***Unboxed to:', dblp_xml)

*** Downloading DBLP file...
***File downloaded: ../data/raw/dblp.xml.gz
***Unboxing file...
***Unboxed to: ../data/raw/dblp.xml


Dane pobrano bezpośrednio ze źródła w formacie skompresowanym .xml.gz (ok. 500MB, po rozpakowanio około 4GB.)
Pobieranie zreazlizowano strumieniowo (stream=True, chunk_size=8192) aby uniknąć ładowania całego pliku do pamięci RAM, plik był zapisywany na dysk batchami po 8KB. Następnie plik rozpakowano do folderu data/raw/ zachowując format XML. 

In [10]:
def parse_xml_to_df(xml_path, min_year):
    records = []
    allowed_tags = {'article', 'inproceedings', 'proceedings', 'book', 'incollection'}
    context = etree.iterparse(xml_path, events=['end'], load_dtd=True)
    for event, element in context:
        if element.tag in allowed_tags:
            year_elem = element.find('year')
            if year_elem is not None and year_elem.text and year_elem.text.isdigit():
                year = int(year_elem.text)
                if year >= min_year:
                    title_elem = element.find('title')
                    title = ''.join(title_elem.itertext()) if title_elem is not None else None
                    authors = [a.text for a in element.findall('author') if a.text]
                    authors_str = ', '.join(authors)
                    source = element.findtext('journal') or element.findtext('booktitle')
                    records.append({
                        'type': element.tag,
                        'year' : year, 
                        'title' : title,
                        'authors': authors_str, 
                        'source' : source
                    })
            element.clear()
            while element.getprevious() is not None:
                del element.getparent()[0]
    return pd.DataFrame(records)

Funkcja parse_xml_to_df() parsuje plik DBLP XML do pandas DataFrame filtrując publikacje po roku. Zastosowano iterparse z biblioteki lxml - parser który czyta plik sewkencyjnie bez ładowania jego całości do RAM. Każdy przetworzony węzeł jest natychmiast usuwany z drzewa (element.clear()) co powoduje stabilne zużycie RAM. Filtrowane są tylko wybrane typy publikacji (article, inproceedings itp.) z określonego przedziału czasowego. 

In [11]:
xml_path = '../data/raw/dblp.xml'
min_year = 2018
df = parse_xml_to_df(xml_path, min_year)

Zdecydowano o ograniczeniu zbioru do publikacji z lat 2018–2025. Rok 2026 został wykluczony ze względu na niewystarczającą liczbę rekordów — dane są niepełne (plik DBLP pobrany w trakcie trwającego roku), co mogłoby zaburzyć analizę trendów.

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3912885 entries, 0 to 3912884
Data columns (total 5 columns):
 #   Column   Dtype
---  ------   -----
 0   type     str  
 1   year     int64
 2   title    str  
 3   authors  str  
 4   source   str  
dtypes: int64(1), str(4)
memory usage: 783.1 MB


In [17]:
df.head()

,type,year,title,authors,source
0,incollection,2021,Blockchains and Distributed Ledgers Uncovered:...,"Eder J. Scheid, Bruno Bastos Rodrigues, Christ...",IFIP's Exciting First 60+ Years
1,incollection,2024,A Common Ground for Developing a Global Consci...,Don Gotterbarn,IFIP TC9 50th Anniversary Anthology
2,incollection,2021,An IFIP WG5.8 State-of-the-Art View on Methods...,"Georg Weichhart, Yves Ducq, Guy Doumeingts",IFIP's Exciting First 60+ Years
3,incollection,2021,IFIP Code of Ethics.,David Kreps,IFIP's Exciting First 60+ Years
4,incollection,2024,Introduction - Legacies of the Formation of IF...,"Christopher Leslie, David Kreps",IFIP TC9 50th Anniversary Anthology


In [18]:
df.to_parquet('../data/processed/dblp_2018.parquet')